# The Compression Floor — grid sweep for DiffuLM @ NeurIPS 2026
### block size × tokens-per-step → repeat-4, boxed rate, correctness, reasoning structure

**What this measures.** dLLMs sell parallel decoding: reveal many tokens per denoising step,
generate faster. This notebook measures **where that breaks**. For each (block size,
tokens-per-step) configuration we generate on a fixed, difficulty-stratified problem set and
record four families of metric:

| metric | what it detects |
|---|---|
| **repeat-4** | degenerate looping (`"the the the"`); 0 = clean, ~1 = total collapse |
| **boxed rate** | did the model reach a `\boxed{}` conclusion at all within budget |
| **correct rate** | was that conclusion right (answer-matched against DeepMath gold) |
| **progress / redundancy / coverage** | *reasoning structure* — does the text move toward the conclusion, loop, or wander |

**Two structural facts that shape this sweep.** First, tokens-per-step cannot exceed block size,
so the grid is **ragged: 13 cells, not 15** (block-4 only supports 1/2/4). Second, forward-pass
cost is `budget / tokens_per_step` — **independent of block size** — because blocks scale down
exactly as sub-steps scale up. So the 1-token/step column dominates the runtime and the block-4
row is nearly free.

**Time-saving techniques used** (measured ~2.5 h → ~40 min on an H100):
1. **Batched generation** across problems — the dominant win (~3×).
2. **Batch compaction** — finished sequences are dropped from the batch, not carried dead.
3. **Three early-stop conditions** — complete `\boxed{}`, EOS, or confirmed collapse
   (repeat-4 > 0.9 after ≥128 tokens). Collapsed configs exit fast instead of grinding to budget.
4. **One model load per block size**, with all its tokens/step cells run before unloading.
5. **Attention mask built once per block**, reused across every sub-step within that block.
6. **Per-cell Drive checkpointing** — the sweep resumes exactly where it stopped.

> Structure metrics are TF-IDF based (sklearn, already in Colab) — deliberately **no new
> dependencies**, since installing an embedding library risks upgrading `transformers` and
> breaking the pinned custom modeling code.


## 1 · GPU & Drive

In [ ]:
!nvidia-smi
import torch, platform
print("\nTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| Python:", platform.python_version())
assert torch.cuda.is_available(), "No GPU — Runtime ▸ Change runtime type ▸ H100."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")
if "H100" not in p.name:
    print("⚠️ Not an H100 — the sweep still runs, just slower. Consider lowering BATCH_SIZE in §3.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2 · Dependencies — torchao removed **before** transformers is imported
`modeling_sdar.py` needs transformers 4.5x. torchao must go first: transformers caches
`is_torchao_available()` at import time, then does a deferred torchao import when loading a model.
**If the transformers version changes, restart the runtime and re-run from the top.**

In [ ]:
import os, re, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
for _n in [n for n in list(sys.modules) if n == "torchao" or n.startswith("torchao.")]:
    del sys.modules[_n]

if not os.path.isdir('/content/dLLM-RL'):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/Gen-Verse/dLLM-RL","/content/dLLM-RL"], check=True)

_FALLBACK = "transformers==4.51.3"
_spec = _FALLBACK
_req = "/content/dLLM-RL/requirements.txt"
if os.path.isfile(_req):
    m = re.search(r"^\s*transformers(\[[^\]]*\])?\s*([=<>!~].*?)\s*(?:#.*)?$", open(_req).read(), re.M)
    if m and m.group(2): _spec = "transformers" + (m.group(1) or "") + m.group(2).strip()
print("Pinning:", _spec)

!pip -q install "{_spec}" "accelerate>=0.33" "datasets>=2.20" sentencepiece packaging

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
print("torchao present?", importlib.util.find_spec("torchao") is not None, "(must be False)")
import transformers; print("transformers:", transformers.__version__)
print("⚠️ If that version just CHANGED: Runtime ▸ Restart session, then run from the top.")


## 3 · Config — the grid, the problem set, and the knobs

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
import os

@dataclass
class GridConfig:
    drive_root: str = "/content/drive/MyDrive/dlm_compression_floor"

    # ---- the grid: block size -> checkpoint. tokens/step is filtered to <= block size. ----
    models: Dict[int, str] = field(default_factory=lambda: {
        4:  "JetLM/SDAR-4B-Chat",       # base = block 4
        16: "JetLM/SDAR-4B-Chat-b16",
        32: "JetLM/SDAR-4B-Chat-b32",
    })
    tokens_per_step: List[int] = field(default_factory=lambda: [1, 2, 4, 8, 16])

    # ---- problem set (fixed across every cell; sampled once, cached to Drive) ----
    dataset_id: str = "zwhe99/DeepMath-103K"
    n_easy: int = 20
    n_medium: int = 10
    n_hard: int = 10
    easy_max: float = 3.0        # difficulty <= easy_max
    hard_min: float = 6.0        # difficulty >  hard_min ; medium is in between
    seed: int = 0

    # ---- generation ----
    budget: int = 768            # max generated tokens per problem
    batch_size: int = 8          # H100-80GB: 8 is safe at this seq len; try 16 to go faster
    greedy: bool = True          # deterministic -> the grid is reproducible

    # ---- early stopping (the big time saver) ----
    stop_on_box: bool = True     # once \boxed{...} closes we have what we need
    stop_on_collapse: bool = True
    collapse_repeat4: float = 0.90
    collapse_min_tokens: int = 128

    # ---- structure analysis ----
    n_chunks: int = 6

cfg = GridConfig()
os.makedirs(cfg.drive_root, exist_ok=True)
RESULTS_PATH = os.path.join(cfg.drive_root, "grid_results.json")
PROBLEMS_PATH = os.path.join(cfg.drive_root, "problem_set.json")

# ragged grid: tokens/step must be <= block size
GRID = [(b, t) for b in sorted(cfg.models) for t in cfg.tokens_per_step if t <= b]
n_prob = cfg.n_easy + cfg.n_medium + cfg.n_hard
fwd = sum(cfg.budget // t for _, t in GRID) * n_prob
print(f"Grid: {len(GRID)} valid cells (tokens/step <= block size)")
for b in sorted(cfg.models):
    print(f"  block {b:>2}: tokens/step {[t for bb,t in GRID if bb==b]}")
print(f"\nProblems: {n_prob} ({cfg.n_easy}E / {cfg.n_medium}M / {cfg.n_hard}H) | budget {cfg.budget}")
print(f"Worst-case forwards: {fwd:,} | batched({cfg.batch_size}) ~{fwd/cfg.batch_size*0.15/60:.0f} min "
      f"before early-stop savings")
print(f"Results -> {RESULTS_PATH}")


## 4 · `flash_attn` → pure-PyTorch replacements
`modeling_sdar.py` hard-imports flash-attn's fused RMSNorm and attention. A meta-path finder
serves numerically-equivalent pure-PyTorch versions (SDPA + hand-written RMSNorm), plus the
`get_imports` patch and the `LossKwargs` rename bridge. No install, no compile.

In [ ]:
import os, sys, types, importlib, importlib.abc, importlib.machinery, importlib.util
import torch, torch.nn.functional as F

USE_FLASH_ATTN = False
for _n in [n for n in list(sys.modules) if n=="flash_attn" or n.startswith("flash_attn.")]:
    del sys.modules[_n]
for _n in [n for n in list(sys.modules) if "modeling_sdar" in n]:
    del sys.modules[_n]

import transformers.dynamic_module_utils as _dmu
_real = getattr(_dmu, "_orig_get_imports", _dmu.get_imports)
_dmu._orig_get_imports = _real
def _pgi(fn):
    imp = list(_real(fn))
    if "flash_attn" in imp and os.path.basename(str(fn)).startswith("modeling_"):
        imp = [i for i in imp if i != "flash_attn"]
    return imp
_dmu.get_imports = _pgi

import transformers.utils as _tu
for _old, _cands in {"LossKwargs": ["TransformersKwargs"]}.items():
    if not hasattr(_tu, _old):
        v = None
        for c in _cands:
            for mp in ("transformers.utils","transformers.processing_utils",
                       "transformers.modeling_utils","transformers"):
                try:
                    m = importlib.import_module(mp)
                    if hasattr(m, c): v = getattr(m, c); break
                except Exception: pass
            if v is not None: break
        setattr(_tu, _old, v if v is not None else type(_old, (dict,), {}))

def _rms_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                 eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                 zero_centered_weight=False, return_dropout_mask=False, out_dtype=None,
                 out=None, residual_out=None):
    xdt = x.dtype
    if x1 is not None: x = x + x1
    base = ((x.float()+residual.float()) if residual_in_fp32 else (x+residual)) if residual is not None \
           else (x.float() if residual_in_fp32 else x)
    nr = base; xf = base.float()
    y = (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(xdt) * \
        ((1.0+weight) if zero_centered_weight else weight)
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _layer_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                   eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                   zero_centered_weight=False, is_rms_norm=False, return_dropout_mask=False,
                   out_dtype=None, out=None, residual_out=None):
    if is_rms_norm:
        return _rms_norm_fn(x, weight, bias, residual, x1, weight1, bias1, eps, dropout_p,
                            rowscale, prenorm, residual_in_fp32, zero_centered_weight,
                            return_dropout_mask, out_dtype, out, residual_out)
    xdt = x.dtype
    if x1 is not None: x = x + x1
    base = ((x.float()+residual.float()) if residual_in_fp32 else (x+residual)) if residual is not None \
           else (x.float() if residual_in_fp32 else x)
    nr = base; xf = base.float(); mu = xf.mean(-1, keepdim=True)
    y = ((xf-mu)*torch.rsqrt((xf-mu).pow(2).mean(-1, keepdim=True)+eps)).to(xdt) * \
        ((1.0+weight) if zero_centered_weight else weight)
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _expand_kv(k, v, nq):
    nk = k.shape[-2]
    if nk != nq:
        r = nq // nk; k = k.repeat_interleave(r, dim=-2); v = v.repeat_interleave(r, dim=-2)
    return k, v

def _flash_attn_func(q,k,v,dropout_p=0.0,softmax_scale=None,causal=False,window_size=(-1,-1),
                     softcap=0.0,alibi_slopes=None,deterministic=False,return_attn_probs=False,**kw):
    k,v = _expand_kv(k,v,q.shape[-2])
    o = F.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),
                                       is_causal=causal, scale=softmax_scale, dropout_p=0.0)
    return o.transpose(1,2)

def _flash_attn_qkvpacked_func(qkv, **kw):
    q,k,v = qkv.unbind(dim=2); return _flash_attn_func(q,k,v,**kw)

def _flash_attn_varlen_func(q,k,v,cu_seqlens_q,cu_seqlens_k,max_seqlen_q=None,max_seqlen_k=None,
                            dropout_p=0.0,softmax_scale=None,causal=False,**kw):
    cq,ck = cu_seqlens_q.tolist(), cu_seqlens_k.tolist(); outs=[]
    for i in range(len(cq)-1):
        qi,ki,vi = q[cq[i]:cq[i+1]], k[ck[i]:ck[i+1]], v[ck[i]:ck[i+1]]
        ki,vi = _expand_kv(ki,vi,qi.shape[-2])
        oi = F.scaled_dot_product_attention(qi.transpose(0,1).unsqueeze(0),
                                            ki.transpose(0,1).unsqueeze(0),
                                            vi.transpose(0,1).unsqueeze(0),
                                            is_causal=causal, scale=softmax_scale, dropout_p=0.0)
        outs.append(oi.squeeze(0).transpose(0,1))
    return torch.cat(outs,0)

def _pad_input(hs, idx, b, s):
    out = hs.new_zeros(b*s, hs.shape[-1]); out[idx] = hs; return out.view(b,s,-1)
def _unpad_input(hs, am, *a, **k):
    sl = am.sum(-1).to(torch.int32); idx = torch.nonzero(am.flatten(), as_tuple=False).flatten()
    h = hs.reshape(-1, hs.shape[-1])[idx]
    cu = torch.zeros(sl.numel()+1, dtype=torch.int32, device=hs.device)
    cu[1:] = torch.cumsum(sl,0); return h, idx, cu, int(sl.max().item())
def _index_first_axis(x, idx): return x.reshape(-1, *x.shape[1:])[idx]

class _RMSNormModule(torch.nn.Module):
    def __init__(self, hidden_size, eps=1e-6, **kw):
        super().__init__(); self.weight = torch.nn.Parameter(torch.ones(hidden_size)); self.eps = eps
    def forward(self, x, residual=None, prenorm=False, **kw):
        return _rms_norm_fn(x, self.weight, None, residual=residual, eps=self.eps, prenorm=prenorm)

_REG = {"rms_norm_fn":_rms_norm_fn, "layer_norm_fn":_layer_norm_fn, "RMSNorm":_RMSNormModule,
        "LayerNorm":torch.nn.LayerNorm, "flash_attn_func":_flash_attn_func,
        "flash_attn_qkvpacked_func":_flash_attn_qkvpacked_func,
        "flash_attn_varlen_func":_flash_attn_varlen_func, "pad_input":_pad_input,
        "unpad_input":_unpad_input, "index_first_axis":_index_first_axis}
def _uns(n):
    def f(*a, **k): raise RuntimeError(f"flash_attn.{n} has no shim but was CALLED — report it.")
    return f
class _FM(types.ModuleType):
    def __getattr__(self, n):
        if n in _REG: return _REG[n]
        if n.startswith("__"): raise AttributeError(n)
        return _uns(n)
class _FF(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fn, path=None, target=None):
        if fn=="flash_attn" or fn.startswith("flash_attn."):
            return importlib.machinery.ModuleSpec(fn, self, is_package=True)
    def create_module(self, spec):
        m=_FM(spec.name); m.__spec__=spec; m.__path__=[]; m.__version__="0.0-shim"; return m
    def exec_module(self, m): pass

try:
    import flash_attn; USE_FLASH_ATTN = True; print("Real flash_attn present.")
except ImportError:
    if not any(isinstance(f,_FF) for f in sys.meta_path): sys.meta_path.insert(0,_FF())
    import flash_attn; print("✅ pure-PyTorch flash_attn replacements active.")

_t = torch.randn(2,4,8); _w = torch.randn(8)
assert torch.allclose(_rms_norm_fn(_t,_w,eps=1e-6),
                      _t*torch.rsqrt(_t.pow(2).mean(-1,keepdim=True)+1e-6)*_w, atol=1e-5)
print("✅ RMSNorm replacement matches reference math.")


## 5 · Metrics
`repeat4` (looping), boxed extraction + answer matching (did it conclude, and correctly), and the
three **structure** metrics. Each is unit-tested inline against synthetic cases so a silent
regression can't quietly corrupt the whole grid.

**Structure metrics, and what each detects.** Split the generation into `n_chunks` ordered pieces:
- **progress** — Spearman correlation between chunk index and that chunk's similarity to the
  reference solution's *conclusion*. Positive = the text moves toward the answer. Negative = it
  drifts away (rambling). ~0 = no directed flow.
- **redundancy** — mean pairwise similarity between chunks. ~1 = saying the same thing over and
  over (the signature of collapse).
- **coverage** — similarity of the whole generation to the whole reference solution: is it even
  on-topic?

In [ ]:
import numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

def repeat4(text: str) -> float:
    """Fraction of 4-grams that are repeats. 0 = no looping, ~1 = total collapse."""
    t = text.split()
    if len(t) < 4: return 0.0
    g = [tuple(t[i:i+4]) for i in range(len(t)-3)]
    return 1.0 - len(set(g))/len(g)

def extract_boxed(text):
    if not text: return None
    i = text.rfind("\\boxed")
    if i == -1: return None
    j = text.find("{", i)
    if j == -1: return None
    d = 0
    for k in range(j, len(text)):
        if text[k] == "{": d += 1
        elif text[k] == "}":
            d -= 1
            if d == 0: return text[j+1:k]
    return None

def has_complete_box(text) -> bool:
    return extract_boxed(text) is not None

def _norm(s):
    if s is None: return None
    s = str(s).strip().replace(" ", "")
    for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")):
        s = s.replace(a,b)
    if s.startswith("\\text{") and s.endswith("}"): s = s[6:-1]
    return s

def answers_match(pred, gold) -> bool:
    a, b = _norm(pred), _norm(gold)
    if a is None or b is None: return False
    if a == b: return True
    try: return abs(float(a)-float(b)) < 1e-6
    except Exception: return False

def structure_metrics(gen: str, ref: str, n_chunks: int = 6):
    """(progress, redundancy, coverage). NaN when the text is too short to chunk."""
    w = gen.split()
    if len(w) < n_chunks*4 or not ref.strip():
        return float("nan"), float("nan"), float("nan")
    chunks = [" ".join(c) for c in np.array_split(np.array(w), n_chunks)]
    rw = ref.split()
    ref_tail = " ".join(rw[-max(20, len(rw)//4):])
    corpus = chunks + [ref_tail, ref, gen]
    try:
        V = TfidfVectorizer(ngram_range=(1,2), min_df=1).fit_transform(corpus)
    except ValueError:
        return float("nan"), float("nan"), float("nan")
    C, tail, full_ref, full_gen = V[:n_chunks], V[n_chunks], V[n_chunks+1], V[n_chunks+2]
    sims = cosine_similarity(C, tail).ravel()
    prog = spearmanr(np.arange(n_chunks), sims).correlation if float(np.std(sims)) > 1e-9 else 0.0
    if prog is None or np.isnan(prog): prog = 0.0
    P = cosine_similarity(C); iu = np.triu_indices(n_chunks, k=1)
    return float(prog), float(P[iu].mean()), float(cosine_similarity(full_gen, full_ref)[0,0])

# ---- inline self-tests ----
assert abs(repeat4("a b c d e f g h") - 0.0) < 1e-9
assert repeat4("a b c d " * 6) > 0.5
assert extract_boxed(r"x \boxed{12}, y \boxed{\frac{1}{2}}") == r"\frac{1}{2}"
assert extract_boxed("no box") is None
assert answers_match(r"\dfrac{1}{2}", r"\frac{1}{2}") and not answers_match("3","4")
_ref = ("First define a1. Next apply the recursion. Then take limits on both sides giving "
        "L = (1/4)^L. Solving the fixed point yields L = 1/2. The final answer is 1/2.")
_p_good,_r_good,_ = structure_metrics(
    "First define a1 and apply the recursion. Next compute a2 and a3. Then observe it converges. "
    "Taking limits gives L = (1/4)^L. Solving the fixed point yields L = 1/2.", _ref)
_p_bad,_r_bad,_ = structure_metrics("the the the the "*12, _ref)
assert _r_bad > 0.9,  f"collapse should show high redundancy, got {_r_bad}"
assert _r_good < 0.5, f"good text should show low redundancy, got {_r_good}"
print(f"✅ metric self-tests pass  (good: prog={_p_good:+.2f} redun={_r_good:.2f} | "
      f"collapsed: prog={_p_bad:+.2f} redun={_r_bad:.2f})")


## 6 · Problem set — difficulty-stratified, sampled once, cached
The **same** problems are used in every grid cell, so differences across the grid are attributable
to the schedule and nothing else. The set is cached to Drive on first run and reloaded thereafter,
which also means a resumed session cannot silently re-sample a different set.

Only problems whose reference solution is self-consistent (its own boxed answer matches
`final_answer`) are kept — the reference is used as the target for the structure metrics, so a
wrong reference would poison them.

In [ ]:
import json, os
from datasets import load_dataset

if os.path.isfile(PROBLEMS_PATH):
    problems = json.load(open(PROBLEMS_PATH))
    print(f"Loaded cached problem set: {len(problems)} problems")
else:
    raw = load_dataset(cfg.dataset_id, split="train").shuffle(seed=cfg.seed)
    diffs = [float(d) for d in raw.select(range(min(4000, len(raw))))["difficulty"] if d is not None]
    print("difficulty percentiles:", {q: round(float(np.percentile(diffs, q)),2)
                                      for q in (5,25,50,75,95)})
    tiers = {"easy": [], "medium": [], "hard": []}
    want  = {"easy": cfg.n_easy, "medium": cfg.n_medium, "hard": cfg.n_hard}
    for ex in raw:
        if all(len(tiers[k]) >= want[k] for k in tiers): break
        d = ex.get("difficulty")
        if d is None: continue
        d = float(d)
        tier = "easy" if d <= cfg.easy_max else ("hard" if d > cfg.hard_min else "medium")
        if len(tiers[tier]) >= want[tier]: continue
        gold = str(ex["final_answer"]); sol = None
        for c in ("r1_solution_1","r1_solution_2","r1_solution_3"):
            cand = ex.get(c)
            if cand and answers_match(extract_boxed(str(cand)), gold): sol = str(cand); break
        if sol is None: continue     # reference must be self-consistent
        tiers[tier].append({"question": ex["question"], "solution": sol,
                            "gold": gold, "difficulty": d, "tier": tier})
    problems = tiers["easy"] + tiers["medium"] + tiers["hard"]
    json.dump(problems, open(PROBLEMS_PATH,"w"))
    print(f"Sampled {len(problems)} problems -> cached to {PROBLEMS_PATH}")

from collections import Counter
print("tiers:", dict(Counter(p["tier"] for p in problems)))
for t in ("easy","medium","hard"):
    ds = [p["difficulty"] for p in problems if p["tier"]==t]
    if ds: print(f"  {t:<7} n={len(ds):>2} difficulty {min(ds):.1f}–{max(ds):.1f}")
if len(problems) < cfg.n_easy + cfg.n_medium + cfg.n_hard:
    print("⚠️ fewer problems than requested — widen easy_max / hard_min in §3, delete the cache, re-run.")


## 7 · Model loading + the block-causal mask

**Why we build the attention mask ourselves.** `modeling_sdar.py`'s outer `forward()` has
`_update_causal_mask` **commented out** — there is no automatic mask construction anywhere in the
model. Whatever `attention_mask` we pass flows unmodified into `SDARAttention` and is used
*directly* as SDPA's `attn_mask`. Passing nothing crashes (`torch.all(None)`); passing all-ones
routes into the "decoding" branch which applies **no mask at all** (valid only for genuine cached
single-token decode, which we never do). So the block structure is entirely our responsibility:
prompt = plain causal, response = block-causal across blocks and **bidirectional within** a block.

**Batching note.** Prompts are **left-padded** so every sequence's response starts at the same
index — which makes `prompt_len` uniform and the block-causal mask shareable across the batch.
Padding is then masked out as *keys* per sample. RoPE is shift-invariant (attention depends on
`i-j`), so a uniform left shift does not change relative positions and needs no `position_ids`
correction. The diagonal is forced True so a fully-padded query row can never produce NaN.

In [ ]:
import torch, glob, shutil, gc, os, re, transformers
from packaging import version as _v
from transformers import AutoTokenizer, AutoModelForCausalLM

_dt_kw = "dtype" if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("4.56.0") \
         else "torch_dtype"

def build_block_causal_mask(seq_len, prompt_len, block_size, device):
    """True = attend. Prompt: plain causal. Response: block-causal across blocks,
    bidirectional within a block (block(i)==block(j) satisfies block(j)<=block(i))."""
    idx = torch.arange(seq_len, device=device)
    is_resp = idx >= prompt_len
    blk = torch.where(is_resp, (idx - prompt_len) // block_size, idx)
    q_resp, k_resp = is_resp.unsqueeze(1), is_resp.unsqueeze(0)
    q_idx, k_idx = idx.unsqueeze(1), idx.unsqueeze(0)
    q_blk, k_blk = blk.unsqueeze(1), blk.unsqueeze(0)
    return ((~q_resp) & (k_idx <= q_idx)) | (q_resp & (~k_resp)) | \
           (q_resp & k_resp & (k_blk <= q_blk))

def _repair_missing_module(model_id, err_msg):
    """Some SDAR repos reference a custom .py they don't ship (e.g.
    fused_linear_diffusion_cross_entropy.py). If another cached checkpoint has it, copy it in."""
    m = re.search(r"file named ([\w\.]+\.py)", str(err_msg))
    if not m: return False
    fname = m.group(1)
    hits = glob.glob(os.path.expanduser(
        f"~/.cache/huggingface/modules/transformers_modules/**/{fname}"), recursive=True)
    if not hits: 
        print(f"    repair: {fname} not found in any cached checkpoint"); return False
    tgt_dirs = glob.glob(os.path.expanduser(
        f"~/.cache/huggingface/modules/transformers_modules/{model_id.replace('/', '/')}/*"))
    if not tgt_dirs:
        print("    repair: target module dir not found"); return False
    dst = os.path.join(sorted(tgt_dirs, key=os.path.getmtime)[-1], fname)
    shutil.copy(hits[0], dst)
    print(f"    repair: copied {fname} -> {dst}")
    return True

def load_model(model_id):
    """Returns (model, tok) or (None, None) on failure. Never raises — one bad checkpoint
    must not kill a multi-cell sweep."""
    for attempt in (1, 2):
        try:
            tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_id, trust_remote_code=True, device_map="cuda",
                attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "sdpa",
                **{_dt_kw: torch.bfloat16})
            model.eval()
            # never take the fused-CE path (it returns logits=None when self.training)
            if hasattr(model.config, "fuse_cross_entropy"):
                model.config.fuse_cross_entropy = False
            if tok.pad_token_id is None: tok.pad_token = tok.eos_token
            tok.padding_side = "left"
            return model, tok
        except Exception as e:
            print(f"    load attempt {attempt} failed: {type(e).__name__}: {str(e)[:120]}")
            if attempt == 1 and _repair_missing_module(model_id, e):
                continue
            return None, None

def free_model():
    """Callers must `del` their own model/tok bindings FIRST — deleting a parameter inside
    a function only unbinds the local name, leaving the caller's reference alive on GPU."""
    gc.collect(); torch.cuda.empty_cache()
    print(f"    VRAM now: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")

print("loader + mask builder ready.")


## 8 · Batched generation engine
One call generates for a whole batch at a given (block size, tokens/step). Per sub-step each
sample independently picks its own top-`tokens_per_step` most-confident **still-masked**
positions — samples do not have to agree on order. Finished sequences are **compacted out of the
batch** so collapsed/early-finishing problems stop costing compute.

In [ ]:
import torch, time

def build_invalid_mask(model, tok):
    """Padded vocab slots + the MASK token itself are never emittable."""
    vs = model.config.vocab_size; rv = len(tok)
    inv = torch.zeros(vs, dtype=torch.bool)
    if rv < vs: inv[rv:] = True
    mid = getattr(tok, "mask_token_id", None) or 151669
    inv[mid] = True
    return inv.to(model.device), mid

def build_student_prompt(tok, q):
    return tok.apply_chat_template(
        [{"role":"user","content": f"{q}\nPlease reason step by step, and put your final answer "
                                   f"within \\boxed{{}}."}],
        tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def batched_generate(model, tok, probs, block_size, tokens_per_step, invalid, mask_id,
                     budget=None, batch_size=None, verbose=False):
    """Returns {problem_index: generated_text}. Greedy (deterministic) when cfg.greedy."""
    budget = budget or cfg.budget
    batch_size = batch_size or cfg.batch_size
    dev = model.device
    out = {}
    order = sorted(range(len(probs)), key=lambda i: len(probs[i]["question"]))  # length-sorted: less padding
    for s in range(0, len(order), batch_size):
        chunk = order[s:s+batch_size]
        prompts = [build_student_prompt(tok, probs[i]["question"]) for i in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True)
        ids = enc.input_ids.to(dev)
        keep_ok = enc.attention_mask.to(dev).bool()          # [B,P] True = real token
        prompt_len = ids.shape[1]
        alive = list(chunk)                                   # original indices still generating
        gen_n = 0
        while gen_n < budget and alive:
            B, cur = ids.shape[0], ids.shape[1]
            work = torch.cat(
                [ids, torch.full((B, block_size), mask_id, dtype=ids.dtype, device=dev)], dim=1)
            L = work.shape[1]
            pos = torch.arange(cur, cur+block_size, device=dev)
            # mask built ONCE per block, reused across every sub-step
            base = build_block_causal_mask(L, prompt_len, block_size, dev)
            pad_ok = torch.cat([keep_ok, torch.ones(B, L-keep_ok.shape[1], dtype=torch.bool, device=dev)], 1)
            m4 = (base.unsqueeze(0) & pad_ok[:, None, :]).unsqueeze(1)      # [B,1,L,L]
            m4 = m4 | torch.eye(L, dtype=torch.bool, device=dev)[None, None]  # no all-masked rows -> no NaN

            still = torch.ones(B, block_size, dtype=torch.bool, device=dev)
            while bool(still.any()):
                logits = model(input_ids=work, attention_mask=m4).logits[:, pos, :].float()
                logits = logits.masked_fill(invalid, float("-inf"))
                lp = torch.log_softmax(logits, dim=-1)
                conf = lp.max(-1).values.masked_fill(~still, float("-inf"))   # [B,bs]
                k = min(tokens_per_step, int(still.sum(1).max().item()))
                topk = conf.topk(k, dim=1).indices                            # [B,k]
                rows = torch.arange(B, device=dev).unsqueeze(1).expand(-1, k)
                valid = still.gather(1, topk)          # a sample may have < k positions left
                choice = logits.argmax(-1).gather(1, topk)
                r, c, v = rows[valid], topk[valid], choice[valid]
                work[r, pos[c]] = v
                still[r, c] = False

            ids = torch.cat([ids, work[:, cur:cur+block_size]], dim=1)
            keep_ok = torch.cat([keep_ok, torch.ones(B, block_size, dtype=torch.bool, device=dev)], 1)
            gen_n += block_size

            # ---- early stopping + batch compaction ----
            finished, keep_rows = [], []
            for bi in range(B):
                g = [t for t in ids[bi, prompt_len:].tolist() if t < len(tok)]
                txt = tok.decode(g, skip_special_tokens=True)
                done = False
                if cfg.stop_on_box and has_complete_box(txt): done = True
                elif tok.eos_token_id is not None and (ids[bi, -block_size:] == tok.eos_token_id).any():
                    done = True
                elif (cfg.stop_on_collapse and gen_n >= cfg.collapse_min_tokens
                      and repeat4(txt) > cfg.collapse_repeat4): done = True
                elif gen_n >= budget: done = True
                if done: out[alive[bi]] = txt; finished.append(bi)
                else: keep_rows.append(bi)
            if finished:
                if not keep_rows: alive = []; break
                kr = torch.tensor(keep_rows, device=dev)
                ids, keep_ok = ids[kr], keep_ok[kr]
                alive = [alive[i] for i in keep_rows]
        for bi, oi in enumerate(alive):        # budget exhausted
            g = [t for t in ids[bi, prompt_len:].tolist() if t < len(tok)]
            out[oi] = tok.decode(g, skip_special_tokens=True)
        if verbose: print(f"    batch {s//batch_size+1}: {len(chunk)} problems done")
    return out

print("batched generation engine ready.")


## 9 · Equivalence check — batching must not change the numbers
Batching is the single largest speedup and also the single largest correctness risk (padding,
per-sample reveal order, mask broadcasting). This cell generates the **same 2 problems** batched
and unbatched and asserts the outputs are identical. If this fails, every number in the grid would
be suspect — so it is a hard assert, not a warning.

In [ ]:
_m, _t = load_model(cfg.models[32])
assert _m is not None, "block-32 checkpoint failed to load — cannot run the equivalence check."
_inv, _mid = build_invalid_mask(_m, _t)
_sub = problems[:2]

_a = batched_generate(_m, _t, _sub, 32, 8, _inv, _mid, budget=128, batch_size=2)
_b = batched_generate(_m, _t, _sub, 32, 8, _inv, _mid, budget=128, batch_size=1)
same = all(_a[i] == _b[i] for i in range(len(_sub)))
for i in range(len(_sub)):
    print(f"  problem {i}: batched=={ 'unbatched' if _a[i]==_b[i] else 'DIFFERS' } "
          f"({len(_a[i].split())} vs {len(_b[i].split())} words)")
assert same, "BATCHED != UNBATCHED — do not trust the sweep. Inspect padding/mask handling."
print("\n✅ batching is exact — the sweep can be trusted.")
del _m, _t, _inv
free_model()


## 10 · Run the grid
One model load per block size; every tokens/step cell for that block size runs before the model is
freed. **Results are written to Drive after every cell**, and completed cells are skipped on
re-run — so an interrupted session resumes with no lost work.

In [ ]:
import json, os, time, gc, numpy as np

results = json.load(open(RESULTS_PATH)) if os.path.isfile(RESULTS_PATH) else {}
def cell_key(b, t): return f"b{b}_t{t}"

def summarize(gen_map, probs):
    rows = []
    for i, p in enumerate(probs):
        txt = gen_map.get(i, "")
        pred = extract_boxed(txt)
        prog, redun, cov = structure_metrics(txt, p["solution"], cfg.n_chunks)
        rows.append({"tier": p["tier"], "n_tokens": len(txt.split()),
                     "repeat4": repeat4(txt), "boxed": pred is not None,
                     "correct": bool(pred is not None and answers_match(pred, p["gold"])),
                     "progress": prog, "redundancy": redun, "coverage": cov})
    def agg(key, sel=None):
        v = [r[key] for r in rows if (sel is None or r["tier"] == sel)]
        v = [x for x in v if not (isinstance(x, float) and np.isnan(x))]
        return float(np.mean(v)) if v else float("nan")
    out = {"n": len(rows)}
    for k in ("repeat4","boxed","correct","progress","redundancy","coverage","n_tokens"):
        out[k] = agg(k)
    for tier in ("easy","medium","hard"):
        out[f"boxed_{tier}"]   = agg("boxed", tier)
        out[f"correct_{tier}"] = agg("correct", tier)
        out[f"repeat4_{tier}"] = agg("repeat4", tier)
    return out, rows

t_start = time.time()
for block_size in sorted(cfg.models):
    todo = [t for b, t in GRID if b == block_size and cell_key(b, t) not in results]
    if not todo:
        print(f"block {block_size}: all cells already done — skipping load."); continue
    print(f"\n{'='*72}\nLoading {cfg.models[block_size]} (block {block_size}) for tokens/step {todo}")
    model, tok = load_model(cfg.models[block_size])
    if model is None:
        print(f"  ❌ could not load — recording skip for block {block_size}")
        for t in todo: results[cell_key(block_size, t)] = {"error": "load_failed"}
        json.dump(results, open(RESULTS_PATH,"w"), indent=1); continue
    invalid, mask_id = build_invalid_mask(model, tok)
    for t in todo:
        c0 = time.time()
        gen_map = batched_generate(model, tok, problems, block_size, t, invalid, mask_id)
        summ, rows = summarize(gen_map, problems)
        summ.update({"block_size": block_size, "tokens_per_step": t,
                     "seconds": time.time()-c0, "per_problem": rows})
        results[cell_key(block_size, t)] = summ
        json.dump(results, open(RESULTS_PATH,"w"), indent=1)
        print(f"  b{block_size:<2} t{t:<2} | {summ['seconds']:6.0f}s | rep4 {summ['repeat4']:.3f} "
              f"| boxed {summ['boxed']:.2f} | correct {summ['correct']:.2f} "
              f"| prog {summ['progress']:+.2f} | redun {summ['redundancy']:.2f}")
    del model, tok, invalid
    free_model()

print(f"\n{'='*72}\nSweep complete in {(time.time()-t_start)/60:.1f} min -> {RESULTS_PATH}")


## 11 · The grid, printed

In [ ]:
import numpy as np, json
results = json.load(open(RESULTS_PATH))
ok = {k:v for k,v in results.items() if "error" not in v}
blocks = sorted({v["block_size"] for v in ok.values()})
tpss   = sorted({v["tokens_per_step"] for v in ok.values()})

def grid_table(metric, fmt="{:.3f}", title=None):
    print(f"\n=== {title or metric} ===")
    hdr = "block \\ tok/step"
    print(hdr.rjust(14) + "".join(f"{t:>9}" for t in tpss))
    for b in blocks:
        row = f"{b:>14}"
        for t in tpss:
            v = ok.get(f"b{b}_t{t}")
            row += f"{'  —':>9}" if v is None else f"{fmt.format(v[metric]):>9}"
        print(row)

grid_table("repeat4",    title="repeat-4  (lower better; ~1 = collapse)")
grid_table("boxed",      title="boxed rate  (reached a conclusion)")
grid_table("correct",    title="correct rate  (conclusion was right)")
grid_table("progress",   fmt="{:+.2f}", title="structure: progress toward conclusion")
grid_table("redundancy", title="structure: redundancy (higher = looping)")
grid_table("coverage",   title="structure: coverage of the reference solution")
grid_table("seconds",    fmt="{:.0f}",  title="wall-clock seconds per cell")

print("\n=== boxed rate by difficulty tier ===")
print(f"{'cell':>10}{'easy':>8}{'medium':>8}{'hard':>8}")
for b in blocks:
    for t in tpss:
        v = ok.get(f"b{b}_t{t}")
        if v: print(f"{'b'+str(b)+' t'+str(t):>10}{v['boxed_easy']:>8.2f}"
                    f"{v['boxed_medium']:>8.2f}{v['boxed_hard']:>8.2f}")


## 12 · Figures

In [ ]:
import matplotlib.pyplot as plt, numpy as np, json, os
plt.rcParams.update({"figure.dpi":120, "font.size":9})
results = json.load(open(RESULTS_PATH))
ok = {k:v for k,v in results.items() if "error" not in v}
blocks = sorted({v["block_size"] for v in ok.values()})
tpss   = sorted({v["tokens_per_step"] for v in ok.values()})

def mat(metric):
    M = np.full((len(blocks), len(tpss)), np.nan)
    for i,b in enumerate(blocks):
        for j,t in enumerate(tpss):
            v = ok.get(f"b{b}_t{t}")
            if v: M[i,j] = v[metric]
    return M

# ---- Fig 1: heatmaps (the money figure) ----
panels = [("repeat4","repeat-4 (collapse)","Reds"), ("boxed","boxed rate","Greens"),
          ("correct","correct rate","Blues"), ("redundancy","redundancy","Oranges")]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax,(m,title,cmap) in zip(axes, panels):
    M = mat(m)
    im = ax.imshow(M, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(tpss)), tpss); ax.set_yticks(range(len(blocks)), blocks)
    ax.set_xlabel("tokens / step"); ax.set_title(title)
    if ax is axes[0]: ax.set_ylabel("block size")
    for i in range(len(blocks)):
        for j in range(len(tpss)):
            if not np.isnan(M[i,j]):
                ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if M[i,j] > 0.55 else "black")
            else:
                ax.text(j, i, "—", ha="center", va="center", fontsize=8, color="grey")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("The compression floor: quality vs. tokens revealed per denoising step", y=1.04)
fig.tight_layout(); fig.savefig(os.path.join(cfg.drive_root,"fig1_heatmaps.png"),
                                bbox_inches="tight"); plt.show()

# ---- Fig 2: the floor as curves ----
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for m, ax, ttl in [("repeat4",axes[0],"repeat-4 ↓ better"),
                   ("boxed",axes[1],"boxed rate ↑ better"),
                   ("progress",axes[2],"progress toward conclusion ↑ better")]:
    for i,b in enumerate(blocks):
        xs = [t for t in tpss if f"b{b}_t{t}" in ok]
        ys = [ok[f"b{b}_t{t}"][m] for t in xs]
        ax.plot(xs, ys, marker="o", label=f"block {b}")
    ax.set_xscale("log", base=2); ax.set_xticks(tpss, tpss)
    ax.set_xlabel("tokens / step"); ax.set_title(ttl); ax.grid(alpha=.3)
axes[0].axhline(0.6, ls="--", c="grey", lw=.8)
axes[0].text(tpss[0], 0.62, "collapse gate", fontsize=7, color="grey")
axes[0].set_ylabel("metric"); axes[0].legend(fontsize=8)
fig.suptitle("Degradation is schedule-driven, not block-size-driven", y=1.04)
fig.tight_layout(); fig.savefig(os.path.join(cfg.drive_root,"fig2_curves.png"),
                                bbox_inches="tight"); plt.show()

# ---- Fig 3: difficulty stratification ----
fig, axes = plt.subplots(1, len(blocks), figsize=(4*len(blocks), 3), squeeze=False)
for ax,b in zip(axes[0], blocks):
    xs = [t for t in tpss if f"b{b}_t{t}" in ok]
    w = 0.25
    for k,(tier,c) in enumerate([("easy","#4c9f70"),("medium","#e0a458"),("hard","#c1666b")]):
        ax.bar([i+(k-1)*w for i in range(len(xs))],
               [ok[f"b{b}_t{t}"][f"boxed_{tier}"] for t in xs], width=w, label=tier, color=c)
    ax.set_xticks(range(len(xs)), xs); ax.set_xlabel("tokens / step")
    ax.set_title(f"block {b}"); ax.set_ylim(0,1); ax.grid(alpha=.3, axis="y")
axes[0][0].set_ylabel("boxed rate"); axes[0][0].legend(fontsize=8)
fig.suptitle("Boxed rate by problem difficulty", y=1.04)
fig.tight_layout(); fig.savefig(os.path.join(cfg.drive_root,"fig3_difficulty.png"),
                                bbox_inches="tight"); plt.show()

# ---- Fig 4: collapse is bimodal, not gradual (per-problem distribution) ----
fig, ax = plt.subplots(figsize=(7,3))
labels, data = [], []
for b in blocks:
    for t in [x for x in tpss if f"b{b}_t{x}" in ok]:
        data.append([r["repeat4"] for r in ok[f"b{b}_t{t}"]["per_problem"]])
        labels.append(f"b{b}\nt{t}")
ax.violinplot(data, showmedians=True)
ax.set_xticks(range(1,len(labels)+1), labels, fontsize=7)
ax.set_ylabel("repeat-4 (per problem)")
ax.set_title("Per-problem repeat-4: collapse is bimodal, not a smooth degradation")
ax.grid(alpha=.3, axis="y")
fig.tight_layout(); fig.savefig(os.path.join(cfg.drive_root,"fig4_bimodality.png"),
                                bbox_inches="tight"); plt.show()
print("figures saved to", cfg.drive_root)


## 13 · Export for the paper

In [ ]:
import csv, json, os
results = json.load(open(RESULTS_PATH))
ok = {k:v for k,v in results.items() if "error" not in v}

agg_csv = os.path.join(cfg.drive_root, "grid_aggregate.csv")
cols = ["block_size","tokens_per_step","n","repeat4","boxed","correct","progress","redundancy",
        "coverage","n_tokens","boxed_easy","boxed_medium","boxed_hard",
        "correct_easy","correct_medium","correct_hard","seconds"]
with open(agg_csv,"w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
    for v in sorted(ok.values(), key=lambda x:(x["block_size"],x["tokens_per_step"])):
        w.writerow({c: v.get(c) for c in cols})

per_csv = os.path.join(cfg.drive_root, "grid_per_problem.csv")
with open(per_csv,"w",newline="") as f:
    w = csv.writer(f)
    w.writerow(["block_size","tokens_per_step","problem_idx","tier","n_tokens",
                "repeat4","boxed","correct","progress","redundancy","coverage"])
    for v in ok.values():
        for i,r in enumerate(v["per_problem"]):
            w.writerow([v["block_size"], v["tokens_per_step"], i, r["tier"], r["n_tokens"],
                        r["repeat4"], int(r["boxed"]), int(r["correct"]),
                        r["progress"], r["redundancy"], r["coverage"]])

print("wrote:\n ", agg_csv, "\n ", per_csv)
skipped = {k:v for k,v in results.items() if "error" in v}
if skipped: print("\n⚠️ cells skipped (checkpoint failed to load):", list(skipped))


## 14 · Reading the results

**The claim this sweep supports.** Generation quality is governed by **tokens revealed per
denoising step**, not by block size. If the columns move together while the rows stay flat, that's
the compression floor: a schedule-level threshold that holds across architectures.

**What each metric contributes.** `repeat-4` and `redundancy` detect collapse (they should agree —
that's a useful internal consistency check). `boxed` separates *"collapsed"* from *"coherent but
didn't finish in budget"* — a distinction raw accuracy hides. `progress` catches the subtle failure
where output is fluent, non-repetitive, on-topic, and still never converges: fluent rambling scores
near zero or negative while real reasoning is positive.

**Fig. 4 matters most for novelty.** If per-problem repeat-4 is bimodal — a cluster near 0 and a
cluster near 1 with little in between — then collapse is a *per-sample phase transition*, not a
gradual quality decline. That framing is stronger than an averages-only story and is not something
the existing literature reports.

**The link to on-policy self-distillation.** d-OPSD trains only on correct generations, which works
because LLaDA-8B-Instruct reaches Pass@1 ≈ 81 / Pass@8 ≈ 96 — nearly every problem yields a usable
trajectory. Wherever this grid shows a boxed rate near zero, that filter has **zero yield**: the
method is inapplicable, not merely worse. The grid is what locates that boundary.

**Before you scale up:** run one cell first (§10 will resume), confirm the numbers look sane, then
let the rest run. And keep §9's equivalence assert green — it is the guarantee that batching didn't
quietly change the results.
